# Plant Disease Classifier: Colab GPU

Use a Colab GPU to train the PlantVillage classifier, save a checkpoint to Google Drive, and optionally expose authenticated GPU inference to the Streamlit app. In Colab, choose Runtime > Change runtime type > T4 GPU before running.

The live tunnel URL is temporary and only works while this notebook runtime stays connected. Add a Colab Secret named `PLANT_DISEASE_API_KEY` before starting the API. Set that same value and the printed tunnel URL in the deployed app's Streamlit Secrets. Never print or commit the API key.

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab, then rerun this cell.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

## Get the project

This public repository is cloned into the Colab runtime. The runtime's preinstalled PyTorch build is kept intact; only the project and API dependencies are installed.

In [ ]:
PROJECT_DIR = Path('/content/plant-disease-classifier')
REPOSITORY_URL = 'https://github.com/afnanmengal/plant-disease-classifier.git'
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'src'))
colab_requirements = PROJECT_DIR / 'requirements-colab.txt'
install_command = [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(colab_requirements)]
subprocess.run(install_command, check=True)
print(f'Project ready: {PROJECT_DIR}')

## Download and train

The public PlantVillage dataset is cached inside Colab. Training uses the same validated loader and CUDA architecture guard as the local project. Adjust `EPOCHS` if you want a longer run.

In [ ]:
import kagglehub
from scripts.download_dataset import find_class_directory
from plant_disease.training import train_model

dataset_cache = kagglehub.dataset_download('emmarex/plantdisease')
DATA_DIR = find_class_directory(dataset_cache)
CHECKPOINT = PROJECT_DIR / 'models' / 'plant_disease_resnet18.pt'
EPOCHS = 5

history = train_model(
    DATA_DIR,
    CHECKPOINT,
    epochs=EPOCHS,
    batch_size=32,
    pretrained=True,
    num_workers=2,
    device='cuda',
)
print(f'Checkpoint: {CHECKPOINT}')
print(history)

## Back up the checkpoint

This copy persists after the Colab runtime is recycled. The live API below still needs an active runtime and the model loaded in GPU memory.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/plant-disease-classifier')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT = DRIVE_DIR / 'plant_disease_resnet18.pt'
shutil.copy2(CHECKPOINT, DRIVE_CHECKPOINT)
print(f'Checkpoint backed up to {DRIVE_CHECKPOINT}')

## Start the temporary GPU API

In the Colab Secrets panel, add `PLANT_DISEASE_API_KEY`. Add the same secret as `colab_api_token` in Streamlit Cloud. The tunnel URL printed below goes in Streamlit Secrets as `colab_api_url`. Keep the URL and API key private.

In [ ]:
from google.colab import userdata

api_key = userdata.get('PLANT_DISEASE_API_KEY')
if not api_key:
    raise RuntimeError('Add PLANT_DISEASE_API_KEY in the Colab Secrets panel first.')

service_env = os.environ.copy()
service_env['PLANT_DISEASE_API_KEY'] = api_key
service_env['PLANT_DISEASE_CHECKPOINT'] = str(CHECKPOINT)
api_log = open('/content/plant-disease-api.log', 'w')
api_command = [
    sys.executable,
    '-m',
    'uvicorn',
    'plant_disease.colab_api:app',
    '--host',
    '127.0.0.1',
    '--port',
    '8000',
]
api_process = subprocess.Popen(
    api_command,
    cwd=PROJECT_DIR,
    env=service_env,
    stdout=api_log,
    stderr=subprocess.STDOUT,
)

import requests
for _ in range(60):
    try:
        health = requests.get('http://127.0.0.1:8000/health', timeout=2)
        health.raise_for_status()
        break
    except requests.RequestException:
        if api_process.poll() is not None:
            raise RuntimeError(Path('/content/plant-disease-api.log').read_text())
        time.sleep(1)
else:
    raise TimeoutError('Colab GPU API did not become ready.')
print(f'GPU API ready: {health.json()}')

In [ ]:
cloudflared_deb = '/tmp/cloudflared.deb'
download_command = (
    'wget -q '
    'https://github.com/cloudflare/cloudflared/releases/latest/download/'
    f'cloudflared-linux-amd64.deb -O {cloudflared_deb} '
    f'&& dpkg -i {cloudflared_deb}'
)
subprocess.run(['bash', '-lc', download_command], check=True)
tunnel_log = open('/content/cloudflared.log', 'w')
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

TUNNEL_URL = None
for _ in range(60):
    log_text = Path('/content/cloudflared.log').read_text(errors='replace')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        TUNNEL_URL = match.group(0)
        break
    if tunnel_process.poll() is not None:
        raise RuntimeError(log_text[-3000:])
    time.sleep(1)
if not TUNNEL_URL:
    raise TimeoutError('Cloudflare tunnel URL did not appear in the log.')
print(f'colab_api_url = {TUNNEL_URL!r}')
print('Set colab_api_token to the same private value stored in Colab Secrets.')
print('Restart this cell after a Colab runtime reset; the URL will change.')